<a href="https://colab.research.google.com/github/MNIKIEMA/differentiable-wonderland-lab-sessions/blob/main/Lab_Merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Guided lab session (PyTorch)

### Model merging fundamentals

This is a short introduction to [model merging](https://huggingface.co/docs/peft/en/developer_guides/model_merging), a set of algorithms to combine multiple pre-trained models into a single model that performs well on all tasks, possibly with no or limited fine-tuning or post-training. As an introduction, we focus on merging two fine-tuned CNN models with task arithmetic methods (although in a practical setting we could focus on zero-shot classification with [CLIP models](https://arxiv.org/abs/2212.04089)).

**Suggested readings before starting**:
- [1] [Merging Models with Fisher-Weighted Averaging](https://arxiv.org/abs/2111.09832) (Matena and Raffel, 2021)
- [2] [Editing models with task arithmetic](https://arxiv.org/abs/2212.04089) (Ilharco et al., 2023)
- [3] [Language Models are Super Mario: Absorbing Abilities from Homologous Models as a Free Lunch](https://openreview.net/forum?id=fq0NaiU8Ex) (Yu et al., 2024)

### VM Setup

In [ ]:
# We use jaxtyping to highlight array shapes when possible
%pip install jaxtyping --quiet
from jaxtyping import Array, Float, Int
from typing import Tuple, List, Callable

In [ ]:
# We use PyTorch Lightning for the fine-tuning (not essential for this lab)
%pip install pytorch_lightning torchmetrics --quiet

### Section 1: Creating the models

We consider two datasets in this lab (MNIST and FashionMNIST). For each dataset, we load a base pre-trained model, and we fine-tune it with a new randomly inizialized classification head. Most of this is very standard and we add it here only for self-completeness.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
# We use a ResNet-18, but you can vary it here
from torchvision.models import resnet18 as model
from torchvision.models import ResNet18_Weights as weights

In [ ]:
WEIGHTS = weights.DEFAULT

> **TODO**: as a warm-up exercise, the function below should return the correct model from torchvision, with the classification head replaced by a new one.

In [ ]:
def init_model(num_classes: int) -> nn.Module:
  # TODO: Complete the code
  pass

In [ ]:
# Sanity check
net = init_model(10)
assert(net(torch.randn(1, 3, 224, 224)).shape == (1, 10))

In [ ]:
# We need to add a greyscale -> RGB conversion to the preprocessing,
# since our images are black and white.
from torchvision.transforms import v2 as T
preprocess = T.Compose([
  T.Grayscale(num_output_channels=3),
  WEIGHTS.transforms()
])

### Interlude: The datasets

We consider a simplified setup where we have two datasets with a compatible number of classes, to avoid having to keep track of multiple classification heads. However, you can easily generalize the notebook to having more than two datasets or datasets with variable number of classes.

In [ ]:
from torchvision import datasets

In [ ]:
# Load the MNIST dataset from torchvision and resize it to ImageNet defaults
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=preprocess)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=preprocess)

In [ ]:
# Load the FashionMNIST dataset
fmnist_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=preprocess)
fmnist_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=preprocess)

In [ ]:
# Build the dataloaders
from torch.utils.data import DataLoader
mnist_train_dataloader = DataLoader(mnist_train, batch_size=64, shuffle=True, num_workers=2)
mnist_test_dataloader = DataLoader(mnist_test, batch_size=64, shuffle=False, num_workers=2)
fmnist_train_dataloader = DataLoader(fmnist_train, batch_size=64, shuffle=True, num_workers=2)
fmnist_test_dataloader = DataLoader(fmnist_test, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
for xb, yb in mnist_train_dataloader:
  print(xb.shape)
  print(yb.shape)
  break

In [ ]:
NUM_CLASSES = len(mnist_train.classes)

### Section 2: Fine-tuning the models

In [ ]:
import pytorch_lightning as pylight
from torchmetrics.classification import MulticlassAccuracy

We use PyTorch Lightning to simplify the fine-tuning. If you are not familiar with it, you can read the [quick introduction](https://lightning.ai/docs/pytorch/stable/starter/introduction.html) on their website, but it is not essential for the lab.

In [ ]:
class Wrapper(pylight.LightningModule):

  def __init__(self, model, num_classes=10):
    super().__init__()
    self.model = model
    self.acc = MulticlassAccuracy(num_classes)

  def forward(self, x):
    return self.model(x)

  def configure_optimizers(self):
    return torch.optim.AdamW(self.model.parameters(), lr=5e-5)

  def training_step(self, batch):
    xb, yb = batch
    y_pred = self(xb)
    loss = F.cross_entropy(y_pred, yb)
    return loss

> **TODO**: Run the cells below to perform the two fine-tunings. Optionally, you can try modifying some hyper-parameters or applying some [advanced training techniques](https://lightning.ai/docs/pytorch/stable/levels/intermediate_level_13.html) to improve slightly the results.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

In [ ]:
# Train the MNIST model
mnist_model = Wrapper(init_model(NUM_CLASSES))
trainer = pylight.Trainer(max_epochs=1, accelerator=DEVICE, precision='bf16-mixed' if DEVICE == 'cpu' else '16-mixed')
trainer.fit(mnist_model, mnist_train_dataloader)

In [ ]:
# Train the FashionMNIST model
fmnist_model = Wrapper(init_model(NUM_CLASSES))
trainer = pylight.Trainer(max_epochs=1, accelerator=DEVICE, precision='bf16-mixed' if DEVICE == 'cpu' else '16-mixed')
trainer.fit(fmnist_model, fmnist_train_dataloader)

In [ ]:
mnist_model = mnist_model.model.eval()
fmnist_model = fmnist_model.model.eval()

In [ ]:
def accuracy(model: nn.Module, dataloader: DataLoader, device) -> float:
  # Utility function to compute accuracy
  import tqdm
  acc = 0
  with torch.no_grad():
    for xb, yb in tqdm.tqdm(dataloader):
      xb, yb = xb.to(device), yb.to(device)
      y_pred = model(xb)
      acc += (y_pred.argmax(dim=1) == yb).sum().item()
  return acc / len(dataloader.dataset)

In [ ]:
def evaluate(model: nn.Module):
  # Run a model on both datasets
  mnist_acc = accuracy(model, mnist_test_dataloader, DEVICE)
  fmnist_acc = accuracy(model, fmnist_test_dataloader, DEVICE)
  print(f'MNIST accuracy: {mnist_acc}')
  print(f'FashionMNIST accuracy: {fmnist_acc}')

In [ ]:
evaluate(mnist_model)
evaluate(fmnist_model)

### Secton 3: Isotropic merging

Denote by $\theta_1$ and $\theta_2$ the weights corresponding to the two fine-tuned models. A basic merging strategy is **isotropic merging** [1], which builds a new model from the average of the original weights:

$$ \theta_{\text{iso}} = \frac{\theta_1 + \theta_2}{2} $$

> **TODO**: Complete the algorithm below to perform isotropic merging. Hint: this is relatively easy by manipulating the models' internal [state dictionaries](https://pytorch.org/tutorials/beginner/saving_loading_models.html#what-is-a-state-dict).

In [ ]:
def isotropic_merge(model1: nn.Module, model2: nn.Module) -> nn.Module:
  # TODO: Complete the code. The weights of the output model should be the average
  # of the weights of model1 and model2 (we assume their architectures are identical).
  pass

In [ ]:
merged_model = isotropic_merge(mnist_model, fmnist_model)
_ = merged_model.eval()

In [ ]:
# Let us see how it performs
evaluate(merged_model)

In [ ]:
del merged_model

> ❓Although we are discussing merging models coming from different fine-tunings, the same applies to other setups. For example, isotropic merging when applied to different runs on the same dataset is called [model soups](https://arxiv.org/abs/2203.05482).

### Interlude: Functional transformations

The previous exercise showed two things: (a) manipulating weights is easy (e.g., sums); but (b) interacting with the `nn.Module` is less elegant, because we need to either re-load the state dictionaries, or overwrite the module's properties. For this reason, model merging is slightly easier in functional frameworks like JAX, where parameters are always handled explicitly. Luckily, PyTorch has its own [JAX-like](https://pytorch.org/docs/stable/func.html) API!

In [ ]:
from torch import func
from torch.utils._pytree import tree_map, PyTree
from functools import partial

In [ ]:
# Extract the parameters and buffers from the model
params = dict(mnist_model.state_dict())

In [ ]:
# Call the model in a functional way - with parameters as input
func.functional_call(mnist_model, params, torch.randn(1, 3, 224, 224))

Note that, in general, we could represent the state of a model as a nested structure of arrays, lists, dictionaries, etc. For handling this, PyTorch has its own implementation of a [pytree](https://github.com/pytorch/pytorch/blob/main/torch/utils/_pytree.py) inspired by JAX, which provides methods for operating on this structure as a tree.

In [ ]:
# Add 1 to all parameters
params = tree_map(lambda x: x + 1, params)

In [ ]:
# Isotropic merging implemented as a tree mapping
params_2 = dict(fmnist_model.state_dict())
merged_params = tree_map(lambda x, y: (x + y) / 2, params, params_2)

In [ ]:
# We can also re-create a callable model by fixing the parameters
merged_model = partial(func.functional_call, model, merged_params)

In [ ]:
evaluate(merged_model)

### Section 4: Task arithmetic (TA)

Consider now the pre-trained model $\theta_0$ and a single fine-tuned model $\theta$. We define the **task vector** of the fine-tuning as:

$$ \tau = \theta - \theta_0 $$

Ideally, this defines a direction in the weight space encoding the task, and we can apply it to any model as:

$$ \theta^\prime = \theta + \alpha \tau $$

For two models, this is simply a rescaled variant of isotropic merging, but for more models, TVs have an interesting arithmetic that allows them to be added, subtracted, etc.

> **TODO**: Consider the MNIST model as base model, and compute the updated parameters with the task vector from FashionMNIST, using `torch.func`.

In [ ]:
def compute_merged_params(base_model: nn.Module, new_model: nn.Module, alpha=0.1) -> PyTree:
  # TODO: Complete the code - use tree_map as above!
  pass

In [ ]:
updated_params = compute_merged_params(mnist_model, fmnist_model)
merged_model = partial(func.functional_call, mnist_model, updated_params)

In [ ]:
evaluate(merged_model)

> **TODO**: Plot the accuracy on MNIST and FashionMNIST as $\alpha$ is varied from 0 to 10, and see how the merge models smoothly interpolates the two extremes!

### Section 5: DARE!

Model merging is a very active topic! In fact, the only thing we need to specify a merging procedure is a way to combine two arrays of the same shape:

$$ \theta_{\text{new}} = \text{merge}(\theta_1, \theta_2) $$

For isotropic merge, $\text{merge}(x, y) = \frac{x + y}{2}$, while for task vectors, $\text{merge}(x, y) = x + \alpha(y - x)$. What other merging strategies can you think of?

> **TODO**: To begin with, let us generalize the previous function in order to make it parametric with respect to the merging function.



In [ ]:
def compute_merged_params_generic(base_model: nn.Module, new_model: nn.Module, merge_fcn: Callable) -> PyTree:
  # TODO: Complete the code
  pass

In [ ]:
assert(compute_merged_params_generic(mnist_model, fmnist_model, lambda x, y: (x + y)/2) == updated_params)

As an example of variant, DARE [3] proposed to randomly drop some values from the task vector before application:

$$ \tau \leftarrow \frac{\tau \odot m}{1 - p} $$

where $m$ is a mask of binary values (having 1 with probability p), and the rescaling is applied to re-normalize the remaining values. Note that this is equivalent to applying a dropout layer to the weights. 🙂

> **TODO**: Implement the DARE merging function and test the result.

In [ ]:
def dare_merge(x, y):
  # TODO: Complete the code.
  pass

In [ ]:
dare_params = partial(compute_merged_params_generic, mnist_model, fmnist_model, dare_merge)
merged_model = partial(func.functional_call, mnist_model, updated_params)

In [ ]:
evaluate(merged_model)